# United States Presidential Debates | Official Corpus Creation

In [1]:
import os
import re
import json
import requests
from bs4 import BeautifulSoup
from collections import OrderedDict

In [2]:

from debate_info import debate_info
from debate_dict import debate_dict

In [3]:
import os
import requests
from bs4 import BeautifulSoup

# The CSS selector to extract the content
CONTENT_SELECTOR = "#content-sm"

# Base directory to save scraped content
BASE_DIR = "all_debates_raw"

# Log file name
LOG_FILE = "scrape_log.txt"

def extract_content_from_url(url, selector):
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    selected_content = soup.select_one(selector)
    return selected_content.get_text() if selected_content else None

def write_content_to_file(directory, filename, content, mode='w'):
    os.makedirs(directory, exist_ok=True)
    with open(os.path.join(directory, filename), mode, encoding='utf8') as file:
        file.write(content)

def log_message(message):
    with open(LOG_FILE, 'a', encoding='utf8') as log_file:
        log_file.write(message + "\n")

def scrape_all_content_and_save(debate_dict):
    for year, data in debate_dict.items():
        raw_directory = os.path.join(BASE_DIR, year, "raw")

        # Scrape the info page
        info_url = data["info"]
        info_content = extract_content_from_url(info_url, CONTENT_SELECTOR)
        if info_content:
            log_message(f"Successfully scraped {info_url}")
            write_content_to_file(raw_directory, "raw-info.txt", info_content)
            log_message(f"Saved info content as {os.path.join(raw_directory, 'raw-info.txt')}")
        else:
            log_message(f"Error scraping {info_url}")
            continue  # Skip to the next year if scraping fails

        # Scrape the debate transcripts
        for i, (date, transcript_url) in enumerate(data["transcripts"], start=1):
            if transcript_url != "Transcript not available.":
                transcript_content = extract_content_from_url(transcript_url, CONTENT_SELECTOR)
                if transcript_content:
                    log_message(f"Successfully scraped {transcript_url}")
                    
                    filename = f"{year}-{date}-debate-raw.txt"
                    file_path = os.path.join(raw_directory, filename)
                    if os.path.isfile(file_path):
                        # If the file exists, append the content
                        write_content_to_file(raw_directory, filename, transcript_content, mode='a')
                    else:
                        # If the file does not exist, create it
                        write_content_to_file(raw_directory, filename, transcript_content)
                    
                    log_message(f"Saved transcript content as {os.path.join(raw_directory, f'{year}-{date}-debate-raw.txt')}")
                else:
                    log_message(f"Error scraping {transcript_url}")

In [4]:
scrape_all_content_and_save(debate_dict)


In [5]:
import os
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize

nltk.download('punkt')

def count_sentences_and_words_in_directory(directory):
    open('scraped_info.txt', 'w').close()  # Clear scraped_info.txt
    count_raw_files = 0
    count_info_files = 0
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.endswith("-raw.txt"):
                count_raw_files += 1
            if file.endswith("-info.txt"):
                count_info_files += 1
            with open(os.path.join(root, file), 'r') as text_file:
                text = text_file.read()
                sentences = sent_tokenize(text)
                words = word_tokenize(text)
                with open('scraped_info.txt', 'a') as out_file:
                    out_file.write(f"{file}\n")
                    out_file.write(f"\ttotal number of sentences: {len(sentences)}\n")
                    out_file.write(f"\ttotal number of words: {len(words)}\n")
    with open('scraped_info.txt', 'a') as out_file:
        out_file.write(f"total number of raw debate transcript files created: {count_raw_files}\n")
        out_file.write(f"total number of raw debate info files created: {count_info_files}\n")

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/pulpdesign/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [6]:
count_sentences_and_words_in_directory("all_debates_raw")